# Lab01 — случайные величины в AqSolDB

**Спринт 1.** Исследование до создания генератора случайных чисел.

> Перед сдачей заполните выделенные места собственными интерпретациями. Код считает статистики, но выводы должны объяснять химический смысл и ограничения данных.

## Навигация
1. [Постановка исследования](#Постановка-исследования)
2. [Загрузка и проверка данных](#Загрузка-и-проверка-данных)
3. [Случайные величины](#Случайные-величины-и-случайный-эксперимент)
4. [EDA и распределения](#EDA-и-распределения)
5. [Связь величин и гипотеза](#Связь-величин-и-гипотеза)
6. [Выборки и оценки](#Выборки-и-оценки)
7. [Выводы](#Выводы-спринта-1)


## Постановка исследования

**Вопрос:** как распределены экспериментальная водная растворимость соединений AqSolDB и их молекулярные дескрипторы, и связана ли растворимость с молекулярной массой и липофильностью?

**Целевая совокупность в этой работе:** записи выбранной очищенной версии AqSolDB. Это позволяет рассуждать о случайном выборе записи из конечного списка. База сформирована из опубликованных источников и не является случайной выборкой всех существующих химических соединений; поэтому обобщение на все вещества требует отдельной оговорки.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 20261020
RNG = np.random.default_rng(SEED)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
DATA_PATH = Path("../data/aqsoldb.csv")


## Загрузка и проверка данных

В AqSolDB поле `Solubility` содержит LogS — десятичный логарифм растворимости в моль/л; `MolWt` — молекулярная масса; `MolLogP` — рассчитанный дескриптор липофильности.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Не найден файл {DATA_PATH.resolve()}. Положите aqsoldb.csv в папку data.")
df_raw = pd.read_csv(DATA_PATH)
print(f"Размер: {df_raw.shape[0]} строк × {df_raw.shape[1]} столбцов")
display(df_raw.head())
display(pd.DataFrame({"тип": df_raw.dtypes.astype(str), "пропусков": df_raw.isna().sum()}))


In [ ]:
# Качество данных и метки источника/надёжности
print("Полностью повторяющихся строк:", df_raw.duplicated().sum())
if "InChIKey" in df_raw:
    print("Повторяющихся InChIKey:", df_raw["InChIKey"].duplicated().sum())
if "Group" in df_raw:
    display(df_raw["Group"].value_counts(dropna=False).rename_axis("Group").to_frame("число записей"))


### Решение об очистке

Пока сохраняем все записи с ненулевыми значениями трёх анализируемых числовых полей. Экстремальное значение само по себе не является основанием для удаления: сначала проверяем его происхождение, запись соединения и метку качества. Если команда решит исключить группу или запись, обоснуйте это здесь и сравните результаты до/после фильтрации.


In [ ]:
required = ["Solubility", "MolWt", "MolLogP"]
missing_cols = sorted(set(required) - set(df_raw.columns))
if missing_cols:
    raise KeyError(f"В файле отсутствуют столбцы: {missing_cols}")
df = df_raw.dropna(subset=required).copy()
df = df.rename(columns={"Solubility": "LogS"})
print(f"Аналитическая таблица: {len(df)} строк; исключено из-за пропусков в ключевых полях: {len(df_raw)-len(df)}")


## Случайные величины и случайный эксперимент

**Случайный эксперимент:** равновероятно выбрать одну запись из конечной аналитической таблицы. Элементарный исход — выбранное соединение.

Для случайного соединения \(\omega\) задаём: \(X(\omega)=\mathrm{LogS}\), \(Y(\omega)=\mathrm{MolWt}\), \(Z(\omega)=\mathrm{MolLogP}\). Величины случайны в этой модели потому, что заранее не знаем, какая запись будет выбрана. Для каждой фиксированной молекулы значения дескрипторов определены её структурой; это не означает, что сами дескрипторы являются случайными при фиксированной структуре.

**Границы модели:** LogS — экспериментально собранное свойство, а MolWt/MolLogP рассчитаны по структуре. Случайность здесь относится к случайному выбору соединения из базы; это не доказательство непредсказуемости химических свойств при известных структуре и условиях.


In [ ]:
variables = {
    "LogS": {"column": "LogS", "meaning": "десятичный логарифм растворимости в воде, mol/L (логарифм безразмерен; в литературе принято обозначение log mol/L)"},
    "MolWt": {"column": "MolWt", "meaning": "молекулярная масса, g/mol"},
    "MolLogP": {"column": "MolLogP", "meaning": "рассчитанный молекулярный logP, безразмерный дескриптор"},
}
for name, meta in variables.items():
    print(f"{name}: столбец {meta['column']}; {meta['meaning']}")


## EDA и распределения

Сначала смотрим на данные без автоматического удаления выбросов. Гистограмма показывает форму распределения, boxplot — медиану, межквартильный размах и крайние наблюдения. Не подменяйте интерпретацию графика утверждением, что данные обязаны иметь нормальное распределение.


In [ ]:
cols = ["LogS", "MolWt", "MolLogP"]
display(df[cols].describe(percentiles=[.25, .5, .75, .9, .95, .99]).T)


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 13))
for i, col in enumerate(cols):
    axes[i, 0].hist(df[col], bins=45, color="#4C78A8", edgecolor="white")
    axes[i, 0].set(title=f"Распределение {col}", xlabel=col, ylabel="Частота")
    axes[i, 1].boxplot(df[col], vert=False, patch_artist=True, boxprops={"facecolor":"#72B7B2"})
    axes[i, 1].set(title=f"Boxplot {col}", xlabel=col)
plt.tight_layout()
plt.show()


In [ ]:
# Асимметрия и эксцесс помогают описать форму, но не заменяют графики
shape = pd.DataFrame({
    "skewness": df[cols].skew(),
    "excess_kurtosis": df[cols].kurtosis(),
    "n": df[cols].count(),
})
display(shape)


### Математическое описание эмпирического распределения

Для конечной совокупности из \(N\) строк распределение любой величины можно задать эмпирической функцией распределения:

\[
F_N(x)=rac{1}{N}\sum_{i=1}^{N}\mathbf{1}(x_i\le x).
\]

Она ставит в соответствие каждому порогу \(x\) долю соединений, у которых значение величины не превышает этот порог. В тексте опишите по графикам, насколько распределение симметрично, есть ли длинные хвосты и отдельные экстремальные значения. Параметрическое семейство выбирайте только после просмотра графиков и обоснования; не называйте распределение нормальным только по форме центральной части.


### Стандартизация

Z-преобразование \(z_i=(x_i-ar{x})/s\) меняет масштаб: среднее становится около 0, стандартное отклонение — около 1. Оно **не превращает** произвольное распределение в нормальное. Используем его для сравнения масштаба величин; физический смысл LogS и MolWt сохраняем в исходных единицах.


In [ ]:
standardized = pd.DataFrame(index=df.index)
for col in cols:
    sd = df[col].std(ddof=0)
    standardized[col + "_z"] = (df[col] - df[col].mean()) / sd
display(standardized.agg(["mean", "std", "min", "max"]).T.round(3))


## Связь величин и гипотеза

**Гипотеза для проверки:** связь между LogS и MolLogP.

\(H_0: ho_s=0\) — коэффициент ранговой корреляции Спирмена в выбранной совокупности равен нулю; \(H_1: ho_s
e0\). При \(lpha=0.05\) значимый ненулевой коэффициент свидетельствует о зависимости величин. Ненулевая корреляция достаточна, чтобы отвергнуть независимость; отсутствие значимой корреляции само по себе независимость не доказывает.


In [ ]:
pairs = [("LogS", "MolWt"), ("LogS", "MolLogP"), ("MolWt", "MolLogP")]
rows = []
for x, y in pairs:
    rho, p = stats.spearmanr(df[x], df[y])
    rows.append({"пара": f"{x} — {y}", "rho_Spearman": rho, "p-value": p})
display(pd.DataFrame(rows).round(5))


In [ ]:
# Диаграммы рассеяния: форма связи и выбросы
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (x, y) in zip(axes, pairs):
    ax.scatter(df[x], df[y], s=9, alpha=.25, color="#4C78A8", edgecolors="none")
    ax.set(xlabel=x, ylabel=y, title=f"{x} и {y}")
plt.tight_layout(); plt.show()


### Интерпретация гипотезы — заполнить после запуска

- Для пары LogS–MolLogP получили \(ho_s=\dots\), \(p=\dots\).
- При выбранном \(lpha\) нулевая гипотеза …
- Направление и сила связи означают …
- Ограничение: статистическая связь в собранной базе не доказывает причинность.


## Выборки и оценки

Считаем всю очищенную таблицу конечной генеральной совокупностью. Без возвращения каждое соединение может попасть в выборку не более одного раза; при возвращении некоторые соединения могут повториться. Поэтому второй способ моделирует независимые извлечения из эмпирического распределения, а первый — выборку из конечной совокупности.


In [ ]:
N = len(df)
n_sample = min(1000, N)
rng = np.random.default_rng(SEED)
sample_without = df.sample(n=n_sample, replace=False, random_state=SEED)
sample_with = df.sample(n=n_sample, replace=True, random_state=SEED)

def metrics(frame, label):
    out = {"набор": label, "n": len(frame)}
    for col in cols:
        out[f"{col}: среднее"] = frame[col].mean()
        out[f"{col}: дисперсия"] = frame[col].var(ddof=0)
    return out

comparison = pd.DataFrame([
    metrics(df, "вся совокупность"),
    metrics(sample_without, "выборка без возвращения"),
    metrics(sample_with, "выборка с возвращением"),
]).set_index("набор")
display(comparison.T.round(4))
print("Уникальных соединений в выборке с возвращением:", sample_with["InChIKey"].nunique(), "из", n_sample)


### Эмпирическая состоятельность оценок

Для каждого размера выборки много раз формируем случайную выборку и отслеживаем ошибку выборочного среднего относительно среднего всей выбранной совокупности. Состоятельность проявляется в уменьшении ошибки/разброса при росте \(n\); одно удачное приближение не является доказательством. Здесь оцениваем конечную таблицу и фиксируем seed для воспроизводимости.


In [ ]:
def convergence(population, sizes, repeats=200, seed=SEED):
    rng = np.random.default_rng(seed)
    N = len(population)
    mu = population.mean()
    records = []
    for n in sizes:
        means_wr = np.empty(repeats)
        means_wor = np.empty(repeats)
        for r in range(repeats):
            means_wr[r] = rng.choice(population, size=n, replace=True).mean()
            means_wor[r] = rng.choice(population, size=n, replace=False).mean()
        for method, estimates in [("с возвращением", means_wr), ("без возвращения", means_wor)]:
            records.append({
                "n": n, "способ": method,
                "смещение": estimates.mean() - mu,
                "RMSE": np.sqrt(np.mean((estimates - mu)**2)),
                "SD оценок": estimates.std(ddof=1),
            })
    return pd.DataFrame(records)

sizes = [10, 30, 100, 300, 1000, 3000, 6000, min(9000, N)]
conv_results = pd.concat([
    convergence(df[col].to_numpy(), sizes, seed=SEED+i)
    for i, col in enumerate(cols)
], keys=cols, names=["величина", "строка"]).reset_index(level=0).rename(columns={"level_0":"величина"})
display(conv_results.head(16).round(5))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, cols):
    sub = conv_results[conv_results["величина"] == col]
    for method, group in sub.groupby("способ"):
        ax.plot(group["n"], group["RMSE"], marker="o", label=method)
    ax.set(xscale="log", yscale="log", xlabel="Размер выборки n", ylabel="RMSE среднего", title=col)
    ax.legend()
plt.tight_layout(); plt.show()


### Вывод по выборкам — заполнить

1. Выборочная средняя и дисперсия отличаются от значений генеральной совокупности на …
2. При каком размере выборки оценка среднего стала достаточно близкой? Критерий близости: …
3. С возвращением и без возвращения разброс оценок …; объяснение …
4. Какой вывод можно сделать о состоятельности выборочной средней на этой конечной совокупности? …


## Выводы спринта 1

Заполните после выполнения ноутбука своими словами:

- какие признаки формы распределений LogS, MolWt и MolLogP обнаружены;
- какие величины оказались зависимыми и по какому статистическому свидетельству;
- подтвердилась ли выбранная гипотеза;
- как выборка отражала генеральные статистики и что изменилось при возвращении;
- на какую совокупность можно переносить результаты и каковы ограничения базы.

## Источники

1. Sorkun, M. C., Khetan, A., Er, S. *AqSolDB, a curated reference set of aqueous solubility and 2D descriptors for a diverse set of compounds*. Scientific Data 6, 143 (2019). https://doi.org/10.1038/s41597-019-0151-1
2. AqSolDB, Harvard Dataverse. https://doi.org/10.7910/DVN/OVHAW8
3. CSV-копия, использованная в проекте: https://github.com/whitead/dmol-book/blob/main/data/curated-solubility-dataset.csv
